# Embryonic integration: CSS HVG3000

Set the paths marked in the code before running this notebook.

Rebuild the selected result from the nine saved stage H5AD files. No project Python modules, TSV or JSON files are required.

The result H5AD stores counts, normalized expression, annotations, measured markers, CSS and the final graph/UMAP. The NPZ model stores the fitted expression reference profiles and projection settings. It can restore CSS without refitting reference clusters.

The default run loads the saved result/model pair and redraws figures. Set `REBUILD_RESULT = True` to run the complete integration. Figure parameters can be edited in the last two cells; set `CHECK_ORIGINAL_STYLE = False` for deliberate style changes.


In [ ]:
#1. =========== Paths and shared settings
# Replace each ROOT placeholder with the required file or directory path before running.
from pathlib import Path
import os, sys

ROOT = Path(".")  # Set the project root before running this template.
# Set the stage input directory; uses integration H5ADs generated by the corresponding fig1_E*_clustering.ipynb notebook.
DATA_DIR = ROOT
# Set the output directory path.
FIGURE_DIR = ROOT
# Set the temporary working directory or file path.
TASK_TMP = ROOT
RANDOM_SEED, N_THREADS, FIGURE_DPI = 0, 4, 600
PREFIX = 'CSS_3000_Embryonic_Inte'
STAGES = ('E9','E10','E11','E13','E14','E15','E16','E17','E18')
STAGE_ORDER = ('E9.5','E10.5','E11.5','E13.5','E14.5','E15.5','E16.5','E17.5','E18.5')

for key, leaf in [('NUMBA_CACHE_DIR','numba'), ('MPLCONFIGDIR','mpl'), ('TMP','temp'), ('TEMP','temp')]:
    # Set the temporary working directory or file path.
    folder = ROOT
    folder.mkdir(parents=True, exist_ok=True)
    os.environ[key] = str(folder)
for key in ('OMP_NUM_THREADS','MKL_NUM_THREADS','OPENBLAS_NUM_THREADS','NUMBA_NUM_THREADS','LOKY_MAX_CPU_COUNT'):
    os.environ[key] = str(N_THREADS)
sys.dont_write_bytecode = True

import hashlib, json, textwrap
from io import BytesIO
from importlib.metadata import version
import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
from scipy import sparse
from scipy.stats import rankdata
from sklearn.preprocessing import scale as sklearn_scale
from threadpoolctl import threadpool_limits
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.lines import Line2D
from PIL import Image as PILImage
from IPython.display import display, Image

sc.settings.verbosity = 0
DATA_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
print({p: version(p) for p in ('scanpy','anndata','numpy','scipy','scikit-learn','umap-learn','igraph')})

def preview(path, width=1200):
    with PILImage.open(path) as picture:
        picture = picture.convert('RGB')
        picture.thumbnail((width, 2200), PILImage.Resampling.LANCZOS)
        buffer = BytesIO()
        picture.save(buffer, format='PNG', optimize=True)
    display(Image(data=buffer.getvalue()))

In [ ]:
#2. =========== Numerical functions and CSS reference model
def array_hash(values):
    values = np.ascontiguousarray(values)
    digest = hashlib.sha256()
    digest.update(str(values.shape).encode())
    digest.update(str(values.dtype).encode())
    digest.update(values.tobytes())
    return digest.hexdigest()

def counts_hash(adata, counts_layer="counts"):
    """Hash counts plus ordered cell/gene identities; never modify the source."""
    matrix = sparse.csr_matrix(adata.layers[counts_layer], copy=True)
    matrix.sum_duplicates()
    matrix.sort_indices()
    digest = hashlib.sha256()
    for names in (adata.obs_names, adata.var_names):
        digest.update(json.dumps(list(map(str, names)), ensure_ascii=False).encode())
    for values in (np.asarray(matrix.shape), matrix.indptr.astype(np.int64),
                   matrix.indices.astype(np.int64), matrix.data):
        digest.update(array_hash(values).encode())
    return digest.hexdigest()

def compute_feature_ranks(a, parameters):
    """Rank on complete-axis log(CP10k); do not filter cells or change counts."""
    HVG_MAX = parameters['n_top_genes']
    HVG_FLAVOR = parameters['flavor']
    STUDY_KEY = parameters['study_key']
    work = a.copy()
    sc.pp.highly_variable_genes(work, n_top_genes=HVG_MAX, flavor=HVG_FLAVOR)
    pooled = work.var.assign(gene=work.var_names).sort_values(
        ['dispersions_norm', 'gene'], ascending=[False, True], kind='stable')
    sc.pp.highly_variable_genes(work, n_top_genes=HVG_MAX,
                              flavor=HVG_FLAVOR, batch_key=STUDY_KEY)
    within = work.var.assign(gene=work.var_names).sort_values(
        ['highly_variable_nbatches', 'dispersions_norm', 'gene'],
        ascending=[False, False, True], kind='stable')
    return {'pooled': pooled, 'within_study': within}

def compute_matched_pca(a, ordered_genes, parameters):
    parameters = dict(parameters)
    assert parameters['mask_var'] is None, 'The exact selected genes must all enter PCA.'
    n_pcs = parameters['n_comps']
    scale_clip = parameters['scale_clip']
    genes = pd.Index(ordered_genes).astype(str)
    assert genes.is_unique and n_pcs < min(len(genes), a.n_obs)
    assert genes.isin(a.var_names).all()
    data = a[:, genes].copy()
    unit_variance = parameters.get('unit_variance', True)
    if unit_variance:
        z = data.X.toarray() if sparse.issparse(data.X) else np.asarray(data.X).copy()
        # Match the original float32 centering and scaling.
        data.X = np.clip((z - z.mean(0)) / np.maximum(z.std(0, ddof=1), 1e-8),
                        -scale_clip, scale_clip)
    else:
        # Keep sparse log(CP10k) values; PCA centers them without scaling or clipping.
        scale_clip = None
    entries = data.X.data if sparse.issparse(data.X) else data.X
    assert np.isfinite(entries).all()
    # Ignore inherited HVG flags after selecting the ranked genes.
    sc.tl.pca(data, n_comps=n_pcs, svd_solver=parameters['svd_solver'],
              random_state=parameters['random_state'], mask_var=parameters['mask_var'])
    assert data.varm['PCs'].shape == (len(genes), n_pcs)
    assert data.uns['pca']['params']['mask_var'] is None
    audit = dict(n_input_genes=len(genes), n_pca_genes=len(genes),
                 pca_gene_ids=genes.tolist(), mask_var=None,
                 n_comps=n_pcs, svd_solver=parameters['svd_solver'],
                 random_state=parameters['random_state'], zero_center=True,
                 unit_variance=unit_variance, scale_clip=scale_clip,
                 pca_loadings_sha256=array_hash(data.varm['PCs']))
    return np.asarray(data.obsm['X_pca'], dtype=np.float32).copy(), audit

def sparse_graph_hash(matrix):
    matrix=sparse.csr_matrix(matrix,copy=True)
    matrix.sum_duplicates();matrix.sort_indices()
    digest=hashlib.sha256(str(matrix.shape).encode())
    for values in (matrix.indptr,matrix.indices,matrix.data):
        digest.update(array_hash(values).encode())
    return digest.hexdigest()

def _css_array_hash(values):
    values = np.ascontiguousarray(values)
    digest = hashlib.sha256()
    digest.update(str(values.shape).encode())
    digest.update(str(values.dtype).encode())
    digest.update(values.tobytes())
    return digest.hexdigest()

def _css_cor_dense(x, y):
    """Official corSparse dense arithmetic, applied to average-tied ranks."""
    n = x.shape[0]
    mu_x = np.ravel(x.mean(0))
    mu_y = np.ravel(y.mean(0))
    covariance = (x.T.dot(y) - n * mu_x[:, None].dot(mu_y[:, None].T)) / (n - 1)
    sd_x = np.ravel(np.sqrt(((x ** 2).sum(0) - n * mu_x ** 2) / (n - 1)))
    sd_y = np.ravel(np.sqrt(((y ** 2).sum(0) - n * mu_y ** 2) / (n - 1)))
    return np.array(covariance / sd_x[:, None].dot(sd_y[:, None].T))

def compute_css_representation(a, ordered_genes, matched_pca, parameters, *, n_threads):
    """Return CSS and real batch-reference evidence; never normalize or relabel.

    Required parameters: batch_key, expected_batch_count, reference_pcs,
    reference_neighbors, reference_resolution, min_reference_clusters,
    random_state, leiden_flavor, leiden_directed, leiden_iterations.
    Dense expression is essential: zeros participate in average tied ranks.
    """
    p = dict(parameters)
    genes = pd.Index(ordered_genes).astype(str)
    pc = np.asarray(matched_pca)
    assert genes.is_unique and genes.isin(a.var_names).all()
    assert pc.shape == (a.n_obs, p['reference_pcs']) and np.isfinite(pc).all()
    assert a.obs[p['batch_key']].notna().all()
    assert a.obs[p['batch_key']].nunique() == p['expected_batch_count']
    expression = a[:, genes].X
    dense = expression.toarray() if sparse.issparse(expression) else np.asarray(expression).copy()
    dense = dense.astype(np.float64)
    assert np.isfinite(dense).all() and dense.min() >= 0
    work = ad.AnnData(X=dense, obs=a.obs[[p['batch_key']]].copy(),
                      var=pd.DataFrame(index=genes))
    work.obsm['X_pca'] = pc.copy()
    batch = work.obs[p['batch_key']].astype(str).to_numpy()
    labels = np.empty(a.n_obs, dtype=object)
    reference_objects, evidence, batch_summary = [], [], []
    with threadpool_limits(limits=n_threads):
        for name in pd.unique(batch):
            ix = np.flatnonzero(batch == name)
            local = ad.AnnData(X=work.X[ix],
                obs=pd.DataFrame(index=work.obs_names[ix]), var=work.var.copy())
            local.obsm['X_pca'] = work.obsm['X_pca'][ix]
            effective_k = min(p['reference_neighbors'], len(ix) - 1)
            assert effective_k >= 2, 'A reference batch is too small to cluster.'
            sc.pp.neighbors(local, use_rep='X_pca', n_neighbors=effective_k,
                            metric='euclidean', random_state=p['random_state'])
            sc.tl.leiden(local, resolution=p['reference_resolution'],
                flavor=p['leiden_flavor'], directed=p['leiden_directed'],
                n_iterations=p['leiden_iterations'], random_state=p['random_state'],
                key_added='reference_cluster')
            cluster = local.obs.reference_cluster.astype(str).to_numpy()
            levels = sorted(pd.unique(cluster), key=int)
            labels[ix] = [name + '|' + c for c in cluster]
            include = len(levels) >= p['min_reference_clusters']
            batch_summary.append(dict(batch=name, n_cells=len(ix), n_clusters=len(levels),
                reference_included=include, effective_neighbors=effective_k))
            if include:
                means = np.vstack([local.X[cluster == c].mean(axis=0) for c in levels])
                names = [name + '|' + c for c in levels]
                reference_objects.append(ad.AnnData(X=means, var=work.var.copy(),
                                                   obs=pd.DataFrame(index=names)))
                evidence.append(dict(batch=name, clusters=names, mean_expression=means.copy()))
        assert reference_objects, 'No batch supplies enough real reference clusters.'
        # Match the original library by aligning sorted intersecting gene IDs.
        aligned = np.intersect1d(work.var_names, reference_objects[0].var_names)
        assert len(aligned) == len(genes)
        ranked_cells = rankdata(work[:, aligned].X.T, method='average', axis=0)
        blocks, undefined = [], []
        for entry, reference in zip(evidence, reference_objects):
            ranked_reference = rankdata(reference[:, aligned].X.T, method='average', axis=0)
            correlation = _css_cor_dense(ranked_cells, ranked_reference)
            invalid = (~np.isfinite(correlation).all(axis=1) |
                       (correlation.std(axis=1, ddof=0) <= 1e-12))
            undefined.append(dict(batch=entry['batch'], undefined_rows=int(invalid.sum())))
            assert not invalid.any(), 'Undefined correlation or batch-wise z scaling.'
            # Use ddof=0, as in the original Python implementation; R uses ddof=1.
            blocks.append(sklearn_scale(correlation, axis=1))
        css = np.concatenate(blocks, axis=1)
    assert css.shape[0] == a.n_obs and np.isfinite(css).all()
    history = dict(method='CSS', parameters=p, n_cells=a.n_obs, n_genes=len(genes),
        n_dimensions=css.shape[1], reference_batches=batch_summary,
        undefined_reference_rows=undefined, dense_average_tied_spearman=True,
        z_ddof=0, css_pca=False, n_threads=n_threads,
        reference_profile_sha256=[_css_array_hash(e['mean_expression']) for e in evidence],
        css_sha256=_css_array_hash(css),
        source_commit='9a81488f4cddbdc1689a8505231a145eab9c3a94',
        caveat='Batches with too few real clusters contribute no reference block; '
               'their cells are retained. No age, identity or region enters CSS.')
    return dict(embedding=css, cell_ids=np.asarray(a.obs_names).astype(str),
                ordered_genes=np.asarray(genes).astype(str), reference_labels=labels,
                reference_profiles=evidence, history=history)

def _validate_css_model(model):
    """Check the complete reference model, including every fitted mean block."""
    assert int(model['format_version']) == 1
    ordered = pd.Index(model['ordered_genes'].astype(str))
    aligned = pd.Index(model['aligned_genes'].astype(str))
    common = pd.Index(model['normalization_gene_ids'].astype(str))
    means = model['reference_means']
    clusters = model['reference_cluster_ids'].astype(str)
    batches = model['reference_batches'].astype(str)
    offsets = model['block_offsets']
    assert ordered.is_unique and common.is_unique and ordered.isin(common).all()
    assert aligned.equals(pd.Index(np.intersect1d(ordered, ordered)))
    assert means.dtype == np.float64 and means.shape == (len(clusters), len(ordered))
    assert np.isfinite(means).all() and (means >= 0).all()
    assert len(set(clusters)) == len(clusters) and len(set(batches)) == len(batches)
    assert offsets.dtype == np.int64 and len(offsets) == len(batches) + 1
    assert offsets[0] == 0 and offsets[-1] == len(clusters) and (np.diff(offsets) > 1).all()
    metadata = json.loads(str(model['metadata_json'].item()))
    history = metadata['history']
    expected = [row for row in history['reference_batches'] if row['reference_included']]
    assert batches.tolist() == [row['batch'] for row in expected]
    assert np.diff(offsets).tolist() == [row['n_clusters'] for row in expected]
    assert means.shape[0] == history['n_dimensions'] and means.shape[1] == history['n_genes']
    hashes = model['profile_sha256'].astype(str).tolist()
    assert hashes == history['reference_profile_sha256']
    for i, (start, stop) in enumerate(zip(offsets[:-1], offsets[1:])):
        assert array_hash(means[start:stop]) == hashes[i]
        assert all(name.startswith(batches[i] + '|') for name in clusters[start:stop])
    assert metadata['normalization']['n_genes'] == len(common)
    assert metadata['normalization']['target_sum'] > 0
    assert metadata['normalization']['gene_axis_sha256'] == array_hash(common.to_numpy(dtype=str))
    assert metadata['projection'] == dict(rank_method='average', correlation='Spearman',
        per_reference_batch_z_ddof=0, reference_dtype='float64', query_dtype='float64',
        missing_features='error; never impute unmeasured genes with zero')
    return metadata

def make_css_model(a, css_result, *, target_sum=10000):
    """Package actual fitted reference means; do not retrain or store a placeholder."""
    genes = pd.Index(css_result['ordered_genes']).astype(str)
    history = css_result['history']
    profiles = css_result['reference_profiles']
    assert np.array_equal(css_result['cell_ids'].astype(str), a.obs_names.astype(str))
    assert len(genes) == history['n_genes'] and genes.isin(a.var_names).all()
    assert a.n_obs == history['n_cells']
    assert len(profiles) == len(history['reference_profile_sha256'])
    assert all(np.asarray(row['mean_expression']).dtype == np.float64 for row in profiles)
    blocks = [np.asarray(row['mean_expression']).copy() for row in profiles]
    offsets = np.r_[0, np.cumsum([len(block) for block in blocks])].astype(np.int64)
    metadata = dict(model_type='CSS_reference_expression', version=1, history=history,
        parameters=dict(history['parameters']),
        normalization=dict(target_sum=target_sum, n_genes=a.n_vars,
            gene_axis_sha256=array_hash(np.asarray(a.var_names, dtype=str)),
            X='float32 log1p(CP10k) on the complete common gene axis before HVG selection',
            counts_layer='counts', scaling_before_reference_means=False),
        projection=dict(rank_method='average', correlation='Spearman',
            per_reference_batch_z_ddof=0, reference_dtype='float64', query_dtype='float64',
            missing_features='error; never impute unmeasured genes with zero'),
        training_cells_sha256=array_hash(np.asarray(a.obs_names, dtype=str)),
        training_counts_sha256=counts_hash(a), training_css_sha256=history['css_sha256'],
        scope='Expression reference model; it does not store a UMAP transform or assign cell types.')
    model = dict(format_version=np.asarray(1, dtype=np.int64),
        ordered_genes=np.asarray(genes, dtype=str),
        aligned_genes=np.asarray(np.intersect1d(genes, genes), dtype=str),
        normalization_gene_ids=np.asarray(a.var_names, dtype=str),
        reference_means=np.vstack(blocks),
        reference_cluster_ids=np.asarray([name for row in profiles for name in row['clusters']], dtype=str),
        reference_batches=np.asarray([row['batch'] for row in profiles], dtype=str),
        block_offsets=offsets,
        profile_sha256=np.asarray(history['reference_profile_sha256'], dtype=str),
        metadata_json=np.asarray(json.dumps(metadata, ensure_ascii=False)))
    _validate_css_model(model)
    return model

def recover_css_model(a, *, target_sum=10000):
    """Recover true reference means from a saved training result without fitting.

    Each recovered block must match the original fit's recorded SHA256. The
    saved source batch and reference-cluster memberships are computational
    cluster identities, not curated cell-type labels.
    """
    history = json.loads(a.uns['css_history_json'])
    genes = pd.Index(np.asarray(a.uns['ordered_genes']).astype(str))
    assert a.n_obs == history['n_cells'] and len(genes) == history['n_genes']
    assert genes.is_unique and genes.isin(a.var_names).all()
    batch = a.obs[history['parameters']['batch_key']].astype(str).to_numpy()
    labels = a.obs['css_reference_cluster'].astype(str).to_numpy()
    assert pd.unique(batch).tolist() == [row['batch'] for row in history['reference_batches']]
    selected = a[:, genes].X
    dense = selected.toarray() if sparse.issparse(selected) else np.asarray(selected).copy()
    dense = dense.astype(np.float64)
    assert np.isfinite(dense).all() and dense.min() >= 0
    profiles = []
    for row in history['reference_batches']:
        ix = np.flatnonzero(batch == row['batch'])
        assert len(ix) == row['n_cells']
        prefix = row['batch'] + '|'
        assert all(value.startswith(prefix) for value in labels[ix])
        levels = sorted(pd.unique(labels[ix]), key=lambda value: int(value[len(prefix):]))
        assert len(levels) == row['n_clusters']
        if row['reference_included']:
            local = dense[ix]
            means = np.vstack([local[labels[ix] == label].mean(axis=0) for label in levels])
            assert array_hash(means) == history['reference_profile_sha256'][len(profiles)]
            profiles.append(dict(batch=row['batch'], clusters=levels, mean_expression=means))
    result = dict(ordered_genes=genes.to_numpy(dtype=str), history=history,
        cell_ids=np.asarray(a.obs_names, dtype=str), reference_profiles=profiles)
    return make_css_model(a, result, target_sum=target_sum)

def project_css_model(a, model, *, n_threads):
    """Project normalized expression to every saved CSS reference dimension.

    Use log1p(CP10k) normalized on normalization_gene_ids; no unit-variance
    scaling, new clustering, batch assignment, or cell-type supervision occurs.
    The complete matrix is projected together to reproduce training arithmetic.
    """
    _validate_css_model(model)
    assert np.array_equal(a.var_names.astype(str), model['normalization_gene_ids'].astype(str))
    assert a.X.dtype == np.float32 and 'log1p' in a.uns, (
        'Normalize raw counts on normalization_gene_ids to float32 log1p(CP10k) before projection.')
    genes = pd.Index(model['ordered_genes'].astype(str))
    aligned = pd.Index(model['aligned_genes'].astype(str))
    selected = a[:, genes].X
    dense = selected.toarray() if sparse.issparse(selected) else np.asarray(selected).copy()
    dense = dense.astype(np.float64)
    assert np.isfinite(dense).all() and dense.min() >= 0
    # Keep the indexing and arithmetic order used to fit the model.
    work = ad.AnnData(X=dense, obs=pd.DataFrame(index=a.obs_names), var=pd.DataFrame(index=genes))
    blocks = []
    with threadpool_limits(limits=n_threads):
        ranked_cells = rankdata(work[:, aligned].X.T, method='average', axis=0)
        for start, stop in zip(model['block_offsets'][:-1], model['block_offsets'][1:]):
            reference = ad.AnnData(X=model['reference_means'][start:stop].copy(),
                                  var=pd.DataFrame(index=genes))
            ranked_reference = rankdata(reference[:, aligned].X.T, method='average', axis=0)
            correlation = _css_cor_dense(ranked_cells, ranked_reference)
            invalid = (~np.isfinite(correlation).all(axis=1) |
                       (correlation.std(axis=1, ddof=0) <= 1e-12))
            assert not invalid.any(), 'Undefined reference correlation or batch-wise z scaling.'
            blocks.append(sklearn_scale(correlation, axis=1))
        embedding = np.concatenate(blocks, axis=1)
    assert embedding.shape == (a.n_obs, len(model['reference_cluster_ids']))
    assert embedding.dtype == np.float64 and np.isfinite(embedding).all()
    return embedding

def validate_css_model_training(a, model, *, n_threads):
    """Require exact reference recovery and exact full training projection."""
    metadata = _validate_css_model(model)
    assert array_hash(np.asarray(a.obs_names, dtype=str)) == metadata['training_cells_sha256']
    assert counts_hash(a) == metadata['training_counts_sha256']
    embedding = project_css_model(a, model, n_threads=n_threads)
    assert embedding.dtype == a.obsm['X_css'].dtype
    assert np.array_equal(embedding, a.obsm['X_css'])
    assert array_hash(embedding) == metadata['training_css_sha256']
    return dict(training_shape=list(embedding.shape), dtype=str(embedding.dtype),
        css_sha256=array_hash(embedding), exact_projection=True,
        reference_batches=len(model['reference_batches']),
        reference_clusters=len(model['reference_cluster_ids']))

def load_css_model(path):
    """Load numeric/Unicode NPZ fields with pickle disabled."""
    with np.load(Path(path), allow_pickle=False) as source:
        model = {key: source[key].copy() for key in source.files}
    _validate_css_model(model)
    return model

def save_css_model(model, path, *, overwrite=False):
    """Save the reference model and verify every field on read-back."""
    _validate_css_model(model)
    path = Path(path)
    assert path.suffix.lower() == '.npz'
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('wb' if overwrite else 'xb') as stream:
        np.savez_compressed(stream, **model)
    restored = load_css_model(path)
    assert set(restored) == set(model)
    for key in model:
        assert restored[key].dtype == model[key].dtype
        assert np.array_equal(restored[key], model[key]), key
    return restored

In [ ]:
#3. =========== Load the saved result or assemble stagewise counts
REBUILD_RESULT = False
# Set the result file path; uses CSS_3000_Embryonic_Inte_adata.h5ad generated by embryonic CSS integration.
RESULT_FILE = ROOT
# Set the model file path; uses CSS_3000_Embryonic_Inte_model.npz generated by embryonic CSS integration.
MODEL_FILE = ROOT
loaded = RESULT_FILE.is_file() and not REBUILD_RESULT

if loaded:
    a = ad.read_h5ad(RESULT_FILE)
    print('Loaded saved CSS result:', a.shape)
else:
    # Set each stage input path; uses {stage}_Embryonic_Inte_adata.h5ad generated by the corresponding fig1_E*_clustering.ipynb notebook.
    parts = [ad.read_h5ad(ROOT) for stage in STAGES]
    for part in parts:
        pd.testing.assert_frame_equal(part.var, parts[0].var)
        assert np.array_equal(part.uns['marker_gene_symbols'], parts[0].uns['marker_gene_symbols'])
        assert np.array_equal(part.uns['marker_groups'], parts[0].uns['marker_groups'])
        values = part.layers['counts'].data
        assert np.isfinite(values).all() and (values >= 0).all() and np.array_equal(values, np.rint(values))
    obs = pd.concat([part.obs for part in parts], axis=0, sort=False)
    assert obs.index.is_unique and obs.atlas_order.is_unique
    positions = np.argsort(obs.atlas_order.to_numpy(), kind='stable')
    counts = sparse.vstack([part.layers['counts'] for part in parts], format='csr', dtype=np.int32)[positions].copy()
    counts.sum_duplicates(); counts.sort_indices(); counts.eliminate_zeros()
    a = ad.AnnData(X=counts.copy(), obs=obs.iloc[positions].copy(), var=parts[0].var.copy(), layers={'counts':counts})
    a.obsm['marker_raw_counts'] = np.concatenate([part.obsm['marker_raw_counts'] for part in parts])[positions]
    a.obsm['marker_measured'] = np.concatenate([part.obsm['marker_measured'] for part in parts])[positions]
    for key in ('marker_gene_symbols','marker_groups','marker_contract_json'):
        a.uns[key] = parts[0].uns[key]
    a.uns['stage_contracts_json'] = json.dumps([json.loads(part.uns['stage_input_contract_json']) for part in parts])
    del parts
assert a.shape == (7355,11467)
assert counts_hash(a) == '55a6737e87849748b4b8b3a24d88c9a5b021e2f62495d9f7765954bfdc52ff1e'
assert a.obs.integration_batch.nunique() == 36 and a.obs.harmony_batch_used.nunique() == 33
assert a.obs.stage_standardized.eq('E10.5').sum() == 73
display(a.obs.stage_standardized.value_counts().reindex(STAGE_ORDER).rename('Cells'))

In [ ]:
#4. =========== Complete-axis normalization, HVG3000 and matched PCA
TARGET_SUM, N_HVG = 10000, 3000
HVG_PARAMETERS = dict(n_top_genes=5000, flavor='seurat', study_key='study_id')
PCA_PARAMETERS = dict(n_comps=30, svd_solver='arpack', random_state=RANDOM_SEED,
    mask_var=None, scale_clip=10, unit_variance=True)

if not loaded:
    a.X = a.layers['counts'].astype(np.float32)
    sc.pp.normalize_total(a, target_sum=TARGET_SUM)
    sc.pp.log1p(a)
    a.uns['X_definition'] = 'log1p(CP10k) on complete common11467 axis, float32 CSR'
    eligible = ~a.var.author_glial_common_excluded936.astype(bool)
    eligible &= ~a.var.gene_symbol.astype(str).str.match(r'(?i)^mt-|^ERCC[-_]')
    with threadpool_limits(limits=N_THREADS):
        rank = compute_feature_ranks(a, HVG_PARAMETERS)['within_study']
        genes = rank.index[eligible.reindex(rank.index)][:N_HVG].astype(str).tolist()
        pc, pca_audit = compute_matched_pca(a, genes, PCA_PARAMETERS)
    a.obsm['X_matched_pca'] = pc
    a.uns['ordered_genes'] = np.asarray(genes, dtype=str)
    a.uns['pca_parameters_json'] = json.dumps(pca_audit)
else:
    genes = a.uns['ordered_genes'].astype(str).tolist()
gene_hash = hashlib.sha256(('\n'.join(genes)+'\n').encode()).hexdigest()
assert len(genes) == N_HVG
assert gene_hash == 'dac8069ab8c5547020b3d09dc795d458c591b337b6959f7ba8c87165469370f5'
print('Features:', len(genes), 'normalization genes:', a.n_vars)

In [ ]:
#5. =========== Fit CSS or load its expression reference model
CSS_PARAMETERS = dict(batch_key='harmony_batch_used', expected_batch_count=33,
    reference_pcs=30, reference_neighbors=15, reference_resolution=.6,
    min_reference_clusters=3, random_state=RANDOM_SEED, leiden_flavor='igraph',
    leiden_directed=False, leiden_iterations=2)
VERIFY_MODEL_PROJECTION = True

if not loaded:
    css = compute_css_representation(a, genes, a.obsm['X_matched_pca'], CSS_PARAMETERS, n_threads=N_THREADS)
    a.obsm['X_css'] = css['embedding'].copy()
    a.obsm['X_rep'] = css['embedding'].copy()
    a.obs['css_reference_cluster'] = pd.Categorical(css['reference_labels'])
    a.uns['css_history_json'] = json.dumps(css['history'])
    a.uns['css_parameters_json'] = json.dumps(CSS_PARAMETERS)
    model = make_css_model(a, css, target_sum=TARGET_SUM)
elif MODEL_FILE.is_file():
    model = load_css_model(MODEL_FILE)
else:
    model = recover_css_model(a, target_sum=TARGET_SUM)
assert np.array_equal(model['ordered_genes'], a.uns['ordered_genes'])
assert a.obsm['X_css'].shape == (7355,97)
assert array_hash(a.obsm['X_css']) == '667da37f16f7904ebf965728b46b7c5170e865bab8e91ed49a4ea2c399c32c0f'
if VERIFY_MODEL_PROJECTION:
    model_validation = validate_css_model_training(a, model, n_threads=N_THREADS)
    print('Model projection: exact; reference clusters:', model_validation['reference_clusters'])

In [ ]:
#6. =========== Neighbors, Leiden and the original UMAP sequence
GRAPH_PARAMETERS = dict(n_neighbors=15, metric='euclidean', resolution=.6,
    flavor='igraph', directed=False, n_iterations=2, random_state=RANDOM_SEED)
UMAP_PARAMETERS = dict(min_dist=.35, spread=1., random_state=RANDOM_SEED,
    init_pos='spectral', alpha=1., gamma=1., negative_sample_rate=5, maxiter=1000)

if not loaded:
    graph = ad.AnnData(obs=a.obs.copy(), var=pd.DataFrame(index=pd.Index([], dtype=str)))
    for key in ('X_rep','X_css','X_matched_pca'):
        graph.obsm[key] = a.obsm[key].copy()
    with threadpool_limits(limits=N_THREADS):
        sc.pp.neighbors(graph, use_rep='X_rep', n_neighbors=GRAPH_PARAMETERS['n_neighbors'],
            metric=GRAPH_PARAMETERS['metric'], random_state=GRAPH_PARAMETERS['random_state'])
        sc.tl.leiden(graph, resolution=GRAPH_PARAMETERS['resolution'], flavor=GRAPH_PARAMETERS['flavor'],
            directed=GRAPH_PARAMETERS['directed'], n_iterations=GRAPH_PARAMETERS['n_iterations'],
            random_state=GRAPH_PARAMETERS['random_state'], key_added='leiden')
        sc.tl.umap(graph, **dict(UMAP_PARAMETERS, maxiter=500))
        assert array_hash(graph.obsm['X_umap']) == '22109adf0e00ec27618f73e786103466950504ae90e5d4832f67619529a70a22'
        sc.tl.umap(graph, **UMAP_PARAMETERS)
    a.obs['leiden'] = graph.obs.leiden.copy()
    a.obsm['X_umap'] = graph.obsm['X_umap'].copy()
    for key in ('connectivities','distances'):
        a.obsp[key] = graph.obsp[key].copy()
    a.uns['neighbors'], a.uns['umap'] = graph.uns['neighbors'].copy(), graph.uns['umap'].copy()
    a.uns['downstream_parameters_json'] = json.dumps(dict(GRAPH_PARAMETERS, **UMAP_PARAMETERS))
    a.uns['method'] = 'CSS'
assert array_hash(a.obsm['X_umap']) == '68f993c4b5338bb742a178295b0f0f5357f5a260403f74cc6d411a8df556beb2'
assert sparse_graph_hash(a.obsp['connectivities']) == 'a9d7ca349576e0ebe4b5470c9c943a918dbe76038d3e6ec455347bdffeffd24b'
assert sparse_graph_hash(a.obsp['distances']) == 'eef55f5a181a6b9d678252309bbe7a83f5120abb7b23ff6a1df1a2547e2dbcac'
print('CSS, graph and UMAP reproduce the original result exactly.')

In [ ]:
#7. =========== Save and read back the result/model pair
def _h5_frame(frame):
    """Normalize object columns for H5AD while retaining missing values."""
    result = frame.copy()
    for key in result.columns:
        column = result[key]
        if isinstance(column.dtype, pd.CategoricalDtype):
            if column.cat.categories.dtype == object:
                categories = list(column.cat.categories)
                if not (all(isinstance(v,str) for v in categories) or
                        all(isinstance(v,(int,float,np.number)) for v in categories)):
                    result[key] = pd.Categorical(column.astype('string'),
                        categories=list(map(str,categories)), ordered=column.cat.ordered)
            continue
        if column.dtype == object:
            present = column.dropna()
            if len(present) and present.map(lambda v:isinstance(v,(bool,np.bool_))).all():
                result[key] = column.astype('boolean')
            elif len(present) and present.map(lambda v:isinstance(v,(int,float,np.number))).all():
                result[key] = pd.to_numeric(column)
            else:
                result[key] = column.astype('string')
    result.index = result.index.astype(str)
    return result

if not loaded:
    a.uns['integration_contract_json'] = json.dumps(dict(n_cells=a.n_obs, n_genes=a.n_vars,
        n_hvg=N_HVG, css_dimensions=97, counts_sha256=counts_hash(a),
        ordered_hvg_sha256=gene_hash, css_sha256=array_hash(a.obsm['X_css']),
        umap_sha256=array_hash(a.obsm['X_umap']), raw_counts_layer='counts',
        normalized_X='float32 log1p(CP10k) on complete11467 axis',
        cell_labels_used_for_fit=False, manual_coordinates=False))
    a.uns['export_provenance_json'] = json.dumps(dict(notebook='embryonic_integration.ipynb',
        input_files=[f'{stage}_Embryonic_Inte_adata.h5ad' for stage in STAGES],
        software_versions={p:version(p) for p in ('scanpy','anndata','numpy','scipy','scikit-learn','umap-learn','igraph')}))
    a.uns['css_model_filename'] = MODEL_FILE.name
    a.obs, a.var = _h5_frame(a.obs), _h5_frame(a.var)
    # Set the temporary working directory or file path.
    temporary = ROOT
    ad.settings.allow_write_nullable_strings = True
    a.write_h5ad(temporary, compression='gzip', convert_strings_to_categoricals=False)
    checked = ad.read_h5ad(temporary)
    pd.testing.assert_frame_equal(checked.obs, a.obs)
    pd.testing.assert_frame_equal(checked.var, a.var)
    assert counts_hash(checked) == counts_hash(a)
    assert np.array_equal(checked.X.data, a.X.data)
    for key in a.obsm:
        assert np.array_equal(checked.obsm[key], a.obsm[key], equal_nan=True)
    for key in ('connectivities','distances'):
        assert sparse_graph_hash(checked.obsp[key]) == sparse_graph_hash(a.obsp[key])
    validate_css_model_training(checked, model, n_threads=N_THREADS)
    os.replace(temporary, RESULT_FILE)
    a = checked
if not loaded or not MODEL_FILE.is_file():
    # Set the temporary working directory or file path.
    temporary = ROOT
    save_css_model(model, temporary, overwrite=True)
    os.replace(temporary, MODEL_FILE)
restored_model = load_css_model(MODEL_FILE)
assert np.array_equal(restored_model['reference_means'], model['reference_means'])
print('Saved result:', RESULT_FILE.name)
print('Saved CSS model:', MODEL_FILE.name)

In [ ]:
#8. =========== Figure functions, category order and colors
UMAP_STYLE = dict(point_size=1.35, point_alpha=.82, label_fontsize=6.75,
    legend_fontsize=6.8, label_stroke=1.45, legend_figsize=(10.3,6.4),
    labels_figsize=(8.2,8.0), connector_distance=12, connector_width=.42, connector_alpha=.70)

DOTPLOT_STYLE = dict(max_dot_area=36, column_gap_ratio=.5, gene_font=6.3,
    row_font=6., row_pitch=.155, gene_rotation=55, color_min=-1., color_max=1., split_max_genes=36)

DISPLAY_TYPES={
 'Otic neurogenic / CVG candidate':'Otic neurogenic candidate',
 'O-2: delaminating otic/CVG neuroblast':'Delaminating otic/CVG neuroblast',
 'O-3: developing CVG neuron (not SGN-specific)':'Developing CVG neuron',
 'Undifferentiated / proliferative CVG':'Proliferative CVG',
 'CVG':'CVG',
 'Undifferentiated SGN':'Undifferentiated SGN',
 'Type Ia + Type II precursor candidate':'Ia/II precursor candidate',
 'Type Ib + Type Ic precursor candidate':'Ib/Ic precursor candidate',
 'Immature Type Ia SGN':'Immature Ia SGN',
 'Immature Type Ib SGN':'Immature Ib SGN',
 'Immature Type Ic SGN':'Immature Ic SGN',
 'Immature Type II SGN':'Immature II SGN'}

GROUP_NAMES={'Transition candidates':'Developmental regulators','Delamination / differentiation':'Delamination','SGN differentiation':'SGN differentiation','Additional synaptic function':'Synaptic function','Additional cell cycle':'Replication','Apex-associated program':'Apex program','Exclusion: VGN candidates':'VGN','Exclusion: geniculate / visceral':'GG / visceral','Exclusion: CNS candidates':'CNS','Exclusion: hair cells':'Hair cells','Exclusion: glia':'Glia','Exclusion: mesenchyme':'Mesenchyme','Exclusion: epithelial':'Epithelium','Exclusion: immune':'Immune','Exclusion: erythroid':'Erythroid'}

CURATED_CATEGORY_ORDER = ['E9.5 | CVG neuroblast', 'E10.5 | O-2: delaminating otic/CVG neuroblast', 'E10.5 | O-3: developing CVG neuron (not SGN-specific)', 'E11.5 | Undifferentiated / proliferative CVG', 'E11.5 | Differentiating SGN', 'E13.5 | Late CVG', 'E13.5 | Differentiating SGN', 'E14.5 | Type Ia + Type II precursor candidate', 'E14.5 | Type Ib + Type Ic precursor candidate', 'E14.5 | Undifferentiated SGN', 'E15.5 | Type Ia + Type II precursor candidate', 'E15.5 | Type Ib + Type Ic precursor candidate', 'E15.5 | Undifferentiated SGN', 'E16.5 | Emerging Type Ic', 'E16.5 | Type Ia + Type II precursor candidate', 'E16.5 | Type Ib + Type Ic precursor candidate', 'E16.5 | Undifferentiated SGN', 'E17.5 | Type Ib + Type Ic precursor candidate', 'E17.5 | Type Ia + Type II precursor candidate', 'E18.5 | Immature Type II SGN', 'E18.5 | Immature Type Ia SGN', 'E18.5 | Immature Type Ib SGN', 'E18.5 | Immature Type Ic SGN', 'E18.5 | Undifferentiated SGN']

CURATED_PALETTE = {'E9.5 | CVG neuroblast': '#b08c43', 'E10.5 | O-2: delaminating otic/CVG neuroblast': '#e08bbd', 'E10.5 | O-3: developing CVG neuron (not SGN-specific)': '#5578e0', 'E11.5 | Undifferentiated / proliferative CVG': '#438cb0', 'E11.5 | Differentiating SGN': '#e08f55', 'E13.5 | Late CVG': '#e05578', 'E13.5 | Differentiating SGN': '#4dc9c9', 'E14.5 | Type Ia + Type II precursor candidate': '#b06d78', 'E14.5 | Type Ib + Type Ic precursor candidate': '#6db08f', 'E14.5 | Undifferentiated SGN': '#79b043', 'E15.5 | Type Ia + Type II precursor candidate': '#8ac97d', 'E15.5 | Type Ib + Type Ic precursor candidate': '#b0946d', 'E15.5 | Undifferentiated SGN': '#d98be0', 'E16.5 | Emerging Type Ic': '#e06c55', 'E16.5 | Type Ia + Type II precursor candidate': '#c94d95', 'E16.5 | Type Ib + Type Ic precursor candidate': '#8b92e0', 'E16.5 | Undifferentiated SGN': '#e0b255', 'E17.5 | Type Ib + Type Ic precursor candidate': '#b0b043', 'E17.5 | Type Ia + Type II precursor candidate': '#e0998b', 'E18.5 | Immature Type II SGN': '#b06743', 'E18.5 | Immature Type Ia SGN': '#9ab06d', 'E18.5 | Immature Type Ib SGN': '#9f65c9', 'E18.5 | Immature Type Ic SGN': '#4dc98b', 'E18.5 | Undifferentiated SGN': '#55bee0'}

DISPLAY_TYPES.update({'CVG neuroblast':'CVG neuroblast', 'Late CVG':'Late CVG',
    'Differentiating SGN':'Differentiating SGN', 'Emerging Type Ic':'Emerging Type Ic'})

def panel_from_adata(adata):
    """Read saved labels and measured markers without changing annotations."""
    required = ('stage_standardized', 'cell_type_plot', 'stage_celltype')
    assert all(key in adata.obs for key in required)
    labels = adata.obs['stage_celltype'].astype(str)
    expected = (adata.obs['stage_standardized'].astype(str) + ' | ' +
                adata.obs['cell_type_plot'].astype(str))
    assert labels.equals(expected), 'Saved stage/type labels are inconsistent'
    assert set(labels) == set(CURATED_CATEGORY_ORDER)
    counts = adata.layers['counts']
    assert np.issubdtype(counts.dtype, np.integer)
    denominator = np.asarray(counts.sum(axis=1)).ravel().astype(np.float64)
    assert (denominator > 0).all()
    raw = np.asarray(adata.obsm['marker_raw_counts'], dtype=np.float64)
    measured = np.asarray(adata.obsm['marker_measured'], dtype=bool)
    assert raw.shape == measured.shape == (adata.n_obs, 133)
    assert np.array_equal(np.isnan(raw), ~measured)
    return dict(cell_ids=np.asarray(adata.obs_names, dtype=str),
        gene_symbols=np.asarray(adata.uns['marker_gene_symbols'], dtype=str),
        marker_groups=np.asarray(adata.uns['marker_groups'], dtype=str),
        raw_counts=raw.copy(), measured=measured.copy(),
        log_cp10k=np.log1p(raw * 10000. / denominator[:, None]),
        common_total_counts=denominator)

def display_category(key,wrapped=False):
 stage,state=key.split(' | ',1)
 name=DISPLAY_TYPES.get(state,state).replace('_',' ')
 return stage+'\n'+textwrap.fill(name,24) if wrapped else stage+' '+name

def umap_pair(b,title,palette,category_order,out_dir,prefix,dpi=600):
 POINT_SIZE,POINT_ALPHA,LABEL_FONTSIZE=(UMAP_STYLE[k] for k in ('point_size','point_alpha','label_fontsize'))
 xy=np.asarray(b.obsm['X_umap']);keys=b.obs.stage_celltype.astype(str).to_numpy();rng=np.random.default_rng(0);order=rng.permutation(len(xy))
 lo=xy.min(0);hi=xy.max(0);pad=np.maximum((hi-lo)*.055,.3);limits=[(lo[i]-pad[i],hi[i]+pad[i]) for i in range(2)]
 result={}
 for mode in ('legend','labels'):
  fig=plt.figure(figsize=UMAP_STYLE['legend_figsize'] if mode=='legend' else UMAP_STYLE['labels_figsize'],facecolor='white');ax=fig.add_axes([.09,.12,.54,.78] if mode=='legend' else [.13,.12,.79,.79])
  ax.scatter(xy[order,0],xy[order,1],c=[palette[g] for g in keys[order]],s=POINT_SIZE,alpha=POINT_ALPHA,linewidths=0)
  ax.set(xlim=limits[0],ylim=limits[1],xlabel='UMAP1',ylabel='UMAP2',title=title);ax.set_box_aspect(1)
  ax.tick_params(labelsize=8);ax.title.set_fontsize(10)
  for spine in ('right','top'):ax.spines[spine].set_visible(False)
  for spine in ('left','bottom'):ax.spines[spine].set_linewidth(1.15)
  if mode=='legend':
   handles=[Line2D([],[],marker='o',color=palette[g],ls='',markersize=5.2,label=f'{display_category(g)} ({np.sum(keys==g)})') for g in category_order]
   legend=ax.legend(handles=handles,loc='center left',bbox_to_anchor=(1.025,.5),fontsize=UMAP_STYLE['legend_fontsize'],frameon=False,handletextpad=.35,labelspacing=.55)
   assert len(legend.texts)==len(category_order)
  else:
   labels=[]
   for key in category_order:
    points=xy[keys==key];center=np.median(points,axis=0);anchor=points[np.argmin(np.sum((points-center)**2,axis=1))]
    text=ax.text(*anchor,display_category(key,True),ha='center',va='center',fontsize=LABEL_FONTSIZE,color='black',linespacing=.96,zorder=5,path_effects=[pe.withStroke(linewidth=UMAP_STYLE['label_stroke'],foreground='white')])
    labels.append((text,anchor.copy()))
   fig.canvas.draw();renderer=fig.canvas.get_renderer();bounds=ax.get_window_extent(renderer)
   accepted=[]
   placement_order=sorted(range(len(labels)),key=lambda i:-np.sum(keys==category_order[i]))
   offsets=[np.array([0.,0.])]+[radius*np.array([np.cos(angle),np.sin(angle)]) for radius in (12,22,35,50,70,95,125) for angle in np.arange(0,2*np.pi,np.pi/8)]
   for i in placement_order:
    t,anchor=labels[i];center=ax.transData.transform(anchor);original=t.get_window_extent(renderer)
    candidates=[]
    for offset in offsets:
     box=original.translated(*offset).expanded(1.045,1.08)
     outside=max(bounds.x0+2-box.x0,0)+max(box.x1-bounds.x1+2,0)+max(bounds.y0+2-box.y0,0)+max(box.y1-bounds.y1+2,0)
     overlaps=sum(box.overlaps(other) for other in accepted)
     candidates.append((1e5*overlaps+1e4*outside+np.linalg.norm(offset),offset,box))
    score,offset,box=min(candidates,key=lambda value:value[0])
    t.set_position(ax.transData.inverted().transform(center+offset));accepted.append(box)
   fig.canvas.draw();renderer=fig.canvas.get_renderer()
   actual=[t.get_window_extent(renderer).expanded(1.01,1.01) for t,_ in labels]
   collisions=sum(actual[i].overlaps(actual[j]) for i in range(len(actual)) for j in range(i))
   if collisions:raise RuntimeError(f'{collisions} overlapping UMAP labels require local position overrides')
   for t,anchor in labels:
    box=t.get_window_extent(renderer);ap=ax.transData.transform(anchor);distance=np.linalg.norm(ap-np.clip(ap,[box.x0,box.y0],[box.x1,box.y1]))
    if distance>UMAP_STYLE['connector_distance']:
     target=np.clip(ap,[box.x0,box.y0],[box.x1,box.y1]);p=ax.transData.inverted().transform(target)
     ax.plot([anchor[0],p[0]],[anchor[1],p[1]],color='0.32',lw=UMAP_STYLE['connector_width'],alpha=UMAP_STYLE['connector_alpha'],zorder=4)
   assert len(labels)==len(category_order)
  fig.canvas.draw();ext=ax.get_window_extent();assert abs(ext.width-ext.height)<.01
  # Set the output file path.
  path=ROOT;fig.savefig(path,dpi=dpi,facecolor='white');plt.close(fig);result[mode]=str(path)
 return result

def stage_cluster_statistics(b, panel):
    """Fixed stage x reviewed identity; measured-only marker denominators."""
    obs = b.obs
    assert np.array_equal(obs.index.astype(str), panel['cell_ids'].astype(str))
    keys = obs['stage_celltype'].astype(str).to_numpy()
    order = [key for key in CURATED_CATEGORY_ORDER if key in set(keys)]
    assert set(order) == set(keys), 'Unreviewed identity must not enter the main plot'
    raw = np.asarray(panel['raw_counts']); log = np.asarray(panel['log_cp10k'])
    measured = np.asarray(panel['measured'], dtype=bool)
    assert raw.shape == log.shape == measured.shape
    assert np.isnan(log[~measured]).all()
    avg=[]; pct=[]; coverage=[]; counts=[]; names=[]
    for key in order:
        mask = keys == key; num = measured[mask].sum(0); n = int(mask.sum())
        avg.append(np.divide(np.where(measured[mask], log[mask], 0).sum(0), num,
            out=np.full(raw.shape[1],np.nan), where=num>0))
        pct.append(np.divide(((raw[mask]>0)&measured[mask]).sum(0)*100, num,
            out=np.full(raw.shape[1],np.nan), where=num>0))
        coverage.append(num/n); counts.append(n)
        names.append(textwrap.fill(display_category(key), 34) + f' (n={n})')
    avg=np.asarray(avg); pct=np.asarray(pct); coverage=np.asarray(coverage)
    scaled=np.full_like(avg,np.nan)
    for j in range(avg.shape[1]):
        valid=np.isfinite(avg[:,j]); v=avg[valid,j]
        if len(v):
            scaled[valid,j]=(v-v.mean())/v.std(ddof=0) if v.std(ddof=0)>0 else 0
    # A one-identity plot cannot carry an across-identity z score.
    mode='mean_log_cp10k' if len(order)==1 else 'gene_z_score'
    values=avg if mode=='mean_log_cp10k' else scaled
    vmax=max(float(np.nanmax(avg)),.01) if mode=='mean_log_cp10k' else 1.
    assert np.array_equal(np.isnan(avg), coverage==0)
    assert np.nanmin(pct)>=0 and np.nanmax(pct)<=100 and sum(counts)==len(obs)
    return dict(row_names=names,row_keys=order,n_cells=counts,mean=avg,pct=pct,
        scaled=scaled,coverage=coverage,color_values=values,color_mode=mode,
        color_min=0. if mode=='mean_log_cp10k' else -1.,color_max=vmax)

def dotplot_figure(stats,panel,indices,title,out_path,dpi=600):
 MAX_DOT_AREA,COLUMN_GAP_RATIO,GENE_FONT,ROW_FONT=(DOTPLOT_STYLE[k] for k in ('max_dot_area','column_gap_ratio','gene_font','row_font'))
 names=panel['gene_symbols'][indices].astype(object).copy(); partial=np.any((stats['coverage'][:,indices]>0)&(stats['coverage'][:,indices]<1),axis=0); names=np.array([str(n)+'†' if p else str(n) for n,p in zip(names,partial)],dtype=object);groups=panel['marker_groups'][indices];rows=stats['row_names'];nr,nc=len(rows),len(names)
 pitch=(1+COLUMN_GAP_RATIO)*np.sqrt(MAX_DOT_AREA)/72;matrix_width=nc*pitch;matrix_height=nr*max(DOTPLOT_STYLE['row_pitch'], .12*max(name.count('\n')+1 for name in rows)+.09)
 left,right,bottom=3.20,.30,1.10;top=2.65+max(0,.045*matrix_height-.30);fw=max(left+matrix_width+right,5.4);fh=bottom+matrix_height+top
 fig=plt.figure(figsize=(fw,fh),facecolor='white');ax=fig.add_axes([left/fw,bottom/fh,matrix_width/fw,matrix_height/fh])
 yy,xx=np.indices((nr,nc));size=stats['pct'][:,indices]*MAX_DOT_AREA/100;values=stats['color_values'][:,indices];valid=np.isfinite(size)&(size>0)
 cmap=LinearSegmentedColormap.from_list('expression_purple',['#F1F1F1','#E8C6DE','#C67ABA','#952E7D','#69005F']);norm=Normalize(stats['color_min'],stats['color_max'],clip=True)
 ax.scatter(xx[valid],yy[valid],s=size[valid],c=values[valid],cmap=cmap,norm=norm,linewidths=0)
 missing=stats['coverage'][:,indices]==0;ax.scatter(xx[missing],yy[missing],s=6,marker='x',color='.77',linewidths=.45)
 ax.set(xlim=(-.5,nc-.5),ylim=(nr-.5,-.5));ax.set_xticks(range(nc),names,rotation=DOTPLOT_STYLE['gene_rotation'],ha='right',fontsize=GENE_FONT);ax.set_yticks(range(nr),rows,fontsize=ROW_FONT)
 ax.tick_params(axis='both',length=0,pad=3);ax.set_frame_on(False)
 # Use the same groups and colors in full and split plots.
 allgroups=list(dict.fromkeys(panel['marker_groups']));pool=['#8A5B78','#4B7E88','#A7773E','#647B4C','#6874A2','#9B6461','#4B8982','#8C7954','#8C6290','#82743D','#56769D','#9A6F88','#7C8E53','#956444','#59698E','#7B6A91','#66826D','#9D7658','#887393','#65878D','#98706A','#828453','#798D9A','#9A775C'];colors={g:pool[i] for i,g in enumerate(allgroups)}
 spans=[]
 for group in dict.fromkeys(groups):
  pos=np.flatnonzero(groups==group);a,b=pos[0],pos[-1];spans.append((group,a,b))
  ax.plot([(a+.10)/nc,(b+.90)/nc],[1.025,1.025],transform=ax.transAxes,color=colors[group],lw=2,clip_on=False)
  short=GROUP_NAMES.get(group,group);ax.text((a+b+1)/(2*nc),1.045,short,rotation=50,ha='center',va='bottom',fontsize=6.8,transform=ax.transAxes)
 legend_y=(bottom+matrix_height+top-.90)/fh
 legend_left=left if fw>=left+5.0+.30 else max(.30,(fw-4.8)/2)
 colorax=fig.add_axes([legend_left/fw,legend_y,2.15/fw,.10/fh]);bar=matplotlib.colorbar.ColorbarBase(colorax,cmap=cmap,norm=norm,orientation='horizontal',ticks=np.linspace(stats['color_min'],stats['color_max'],5));bar.outline.set_visible(False);bar.ax.tick_params(labelsize=6,length=2,pad=1);bar.ax.set_title('Mean log1p(CP10k)' if stats['color_mode']=='mean_log_cp10k' else 'Avg. Exp. (gene-wise z score)',fontsize=7,pad=3)
 sizeleft=legend_left+2.65;sizewidth=2.15;sizeax=fig.add_axes([sizeleft/fw,legend_y,sizewidth/fw,.29/fh]);sizeax.set(xlim=(-.6,4.6),ylim=(-.7,.7));sizeax.axis('off')
 for i,p in enumerate([0,25,50,75,100]):
  if p:sizeax.scatter(i,.05,s=p*MAX_DOT_AREA/100,color='#952E7D',linewidths=0)
  sizeax.text(i,-.48,str(p),ha='center',fontsize=6)
 sizeax.text(2,.6,'% Exp.',ha='center',fontsize=7)
 # Wrap titles using measured text bounds and the available width.
 title_width=max(24,int((fw-left-.30)*72/(9*.50)))
 title_artist=fig.text(left/fw,1-.18/fh,textwrap.fill(title,title_width),ha='left',va='top',fontsize=9)
 fig.canvas.draw();title_renderer=fig.canvas.get_renderer()
 right_limit=fig.bbox.x1-.30*fig.dpi
 while title_artist.get_window_extent(title_renderer).x1>right_limit and title_width>12:
  title_width-=1;title_artist.set_text(textwrap.fill(title,title_width))
  fig.canvas.draw();title_renderer=fig.canvas.get_renderer()
 title_box=title_artist.get_window_extent(title_renderer)
 assert title_box.x1<=right_limit+.01, 'Figure title exceeds its measured right margin'
 legend_top=max(colorax.title.get_window_extent(title_renderer).y1,max(t.get_window_extent(title_renderer).y1 for t in sizeax.texts))
 assert title_box.y0>=legend_top+3, 'Figure title overlaps a top legend; shorten the title'
 fig.text(.25/fw,.10/fh,'x: unmeasured; blank: measured zero. †: partially measured (including Cpne9); statistics use measured cells only.\nRows: fixed stage x reviewed identity, matching UMAP counts. Fixed-annotation DotPlots do not assess integration quality.\nSingle-identity panels show mean log1p(CP10k); all split panels retain the full-panel color scale.',ha='left',va='bottom',fontsize=5.7,color='.30')
 fig.canvas.draw();r=fig.canvas.get_renderer();column_px=ax.get_window_extent(r).width/nc;diameter_px=np.sqrt(MAX_DOT_AREA)*fig.dpi/72
 assert abs(column_px/diameter_px-(1+COLUMN_GAP_RATIO))<1e-4
 # Check that full and split plots use the same statistics.
 assert np.allclose(size,stats['pct'][:,indices]*MAX_DOT_AREA/100,equal_nan=True)
 fig.savefig(out_path,dpi=dpi,facecolor='white');plt.close(fig)

def dotplot_set(b,panel,title,out_dir,prefix,dpi=600):
 title = ('Fixed reviewed identities | measured log1p(CP10k)' if b.n_obs==7355 else title)
 stats=stage_cluster_statistics(b,panel);n=len(panel['gene_symbols']);paths=[]
 # Set the output file path.
 path=ROOT;dotplot_figure(stats,panel,np.arange(n),title,path,dpi);paths.append(str(path))
 # Split only at marker-group boundaries, preserving all rows and encodings.
 blocks=[];current=[]
 for group in dict.fromkeys(panel['marker_groups']):
  ix=np.flatnonzero(panel['marker_groups']==group).tolist()
  if len(current)+len(ix)>DOTPLOT_STYLE['split_max_genes'] and current:blocks.append(current);current=[]
  current.extend(ix)
 if current:blocks.append(current)
 for i,ix in enumerate(blocks,1):
  # Set the output file path.
  path=ROOT;dotplot_figure(stats,panel,np.array(ix),title,path,dpi);paths.append(str(path))
 return paths,stats

In [ ]:
#9. =========== UMAP legend and label versions
UMAP_POINT_SIZE, UMAP_ALPHA = 1.35, .82
CHECK_ORIGINAL_STYLE = True  # Set False for deliberate appearance changes.
UMAP_STYLE.update(point_size=UMAP_POINT_SIZE, point_alpha=UMAP_ALPHA)
title = 'CSS HVG3000 | all 97 CSS dimensions | UMAP 1000 epochs'
title += '\n7,355 reviewed embryonic cells; unchanged k15 graph'
paths = umap_pair(a, title, CURATED_PALETTE, CURATED_CATEGORY_ORDER, FIGURE_DIR, PREFIX, dpi=FIGURE_DPI)
expected_png_sha256 = {'legend': '2a1d0e523f1b6c0abec3d722afabbc255b5a82ab1000e3ed8e47a5607f201d4d', 'labels': 'e041966c96415335dd222f3ff31a0244f1835f7f8b62b0ed08b68ae4645f82fc'}
for mode in ('legend','labels'):
    if CHECK_ORIGINAL_STYLE:
        assert hashlib.sha256(Path(paths[mode]).read_bytes()).hexdigest() == expected_png_sha256[mode]
    preview(paths[mode], width=1000)
print('Both UMAPs saved; original-style verification:', CHECK_ORIGINAL_STYLE)

In [ ]:
#10. =========== Full marker DotPlot and four split panels
MAX_DOT_AREA, COLUMN_GAP_RATIO = 36, .5
CHECK_ORIGINAL_STYLE = True  # Set False for deliberate appearance changes.
DOTPLOT_STYLE.update(max_dot_area=MAX_DOT_AREA, column_gap_ratio=COLUMN_GAP_RATIO)
panel = panel_from_adata(a)
dot_paths, stats = dotplot_set(a, panel, 'Fixed reviewed identities | measured log1p(CP10k)', FIGURE_DIR, PREFIX, dpi=FIGURE_DPI)
assert len(dot_paths) == 5 and sum(stats['n_cells']) == a.n_obs
expected_png_sha256 = {'full': '3be7d34e613067bd03444b6029880bbb0b6a1b6f76f516cacb74cba3cb2a757a', '1': '308de465f78c0a6a715eab5f4b60f4f31382cd7f15aad6ec31fe5c28c9814713', '2': '4547fac21b3d1c390ec8d81cbe3d00fa8bc45532d7f9a4cf7d069c055b260daa', '3': '013132ac78960029107198f85a1df5a2b9a85eaee1447f48c3c8fd19546ca5ad', '4': 'e9ab0c16381c99cb880383cc7d9db77d999db3755ffa6eb0d04572c881f90c07'}
for suffix, path in zip(('full','1','2','3','4'), dot_paths):
    if CHECK_ORIGINAL_STYLE:
        assert hashlib.sha256(Path(path).read_bytes()).hexdigest() == expected_png_sha256[suffix]
    preview(path)
print('Five DotPlots saved; unmeasured markers remain NaN.')

# modify figures

In [ ]:
#1. =========== Shared final UMAP settings
from pathlib import Path
import ast
import json
import sys
import numpy as np
import anndata as ad
import matplotlib.pyplot as plt
from matplotlib import font_manager, colors as mcolors
from matplotlib.lines import Line2D
import textwrap

try:
    from reportlab.pdfgen import canvas
except ModuleNotFoundError:
    
    from reportlab.pdfgen import canvas
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont

get_ipython().run_line_magic('matplotlib', 'inline')
ROOT = Path(".")  # Set the project root before running this template.
# Set the stage input directory; uses integration H5ADs generated by the corresponding fig1_E*_clustering.ipynb notebook.
DATA_DIR = ROOT
# Set the output directory path.
OUTPUT_DIR = ROOT
# Set the color notebook path; uses fig1_Whole_integration.ipynb from whole-period integration.
COLOR_NOTEBOOK = ROOT
PREFIX = 'CSS_3000_Embryonic_Inte'
# Set the Arial font file path.
ARIAL_FILE = ROOT

PLOT_SIZE = 5.6          # Main axes width and height, in inches
MARGINS = (0.80, 0.75, 1.85, 0.65)  # Left, bottom, right, top, in inches
POINT_SIZE, POINT_ALPHA = 1.35, 0.82
TITLE_SIZE, TITLE_PAD = 10, 10
AXIS_LABEL_SIZE, AXIS_LABEL_PAD = 9, 5
TICK_SIZE, TICK_PAD, TICK_LENGTH = 7, 3, 3
LABEL_SIZE, LABEL_WRAP = 6.75, 24
LEGEND_SIZE, LEGEND_MARKER_SIZE = 7.5, 4.5
AXIS_WIDTH = 1.1
RANDOM_SEED = 0          # Preserves the original point drawing order
LABEL_OFFSETS_PT = {
    'E9.5 | CVG neuroblast': (-9.184402, 22.173109),
    'E10.5 | O-2: delaminating otic/CVG neuroblast': (0, 6),
    'E10.5 | O-3: developing CVG neuron (not SGN-specific)': (-22.627417, 22.627417),
    'E11.5 | Differentiating SGN': (-4.242641, -4.242641),
    'E13.5 | Differentiating SGN': (9.184402, 22.173109),
    'E14.5 | Type Ib + Type Ic precursor candidate': (18, 0),
    'E15.5 | Type Ia + Type II precursor candidate': (8.485281, -8.485281),
    'E15.5 | Type Ib + Type Ic precursor candidate': (-38.802940, -16.072704),
    'E15.5 | Undifferentiated SGN': (-2.296101, -5.543277),
    'E17.5 | Type Ia + Type II precursor candidate': (29.698485, -29.698485),
    'E17.5 | Type Ib + Type Ic precursor candidate': (-12.727922, -12.727922),
    'E18.5 | Immature Type II SGN': (11.086554, 4.592201),
}                       # Label offsets in points; cell coordinates stay unchanged

font_manager.fontManager.addfont(str(ARIAL_FILE))
pdfmetrics.registerFont(TTFont('Arial', str(ARIAL_FILE)))
plt.rcParams.update({'font.family': 'Arial', 'text.usetex': False,
                     'axes.grid': False, 'figure.facecolor': 'white'})

#2. =========== Read the saved CSS result and shared literal colors
# Set the result file path; uses CSS_3000_Embryonic_Inte_adata.h5ad generated by embryonic CSS integration.
a = ad.read_h5ad(ROOT)
# Set the model file path; uses CSS_3000_Embryonic_Inte_model.npz generated by embryonic CSS integration.
with np.load(ROOT, allow_pickle=False) as saved:
    model = {key: saved[key].copy() for key in saved.files}
assert a.shape == (7355, 11467)
assert model['reference_means'].shape == (97, 3000)
assert np.array_equal(a.uns['ordered_genes'], model['ordered_genes'])
assert np.array_equal(a.var_names.astype(str), model['normalization_gene_ids'])
xy = np.asarray(a.obsm['X_umap']).copy()
annotation_keys = a.obs['stage_celltype'].astype(str).to_numpy()
stage = a.obs['stage_standardized'].astype(str).to_numpy()
assert xy.shape == (a.n_obs, 2) and np.isfinite(xy).all()
assert np.array_equal(annotation_keys, stage + ' | ' + a.obs['cell_type_plot'].astype(str).to_numpy())

shared_colors = {}
color_names = {'CELL_ANNOTATION_COLORS', 'DEVELOPMENTAL_STAGE_COLORS'}
for cell in json.loads(COLOR_NOTEBOOK.read_text(encoding='utf-8'))['cells']:
    source = ''.join(cell['source'])
    if cell['cell_type'] != 'code' or not any(name in source for name in color_names):
        continue
    for node in ast.parse(source).body:
        if isinstance(node, ast.Assign):
            for target in node.targets:
                if isinstance(target, ast.Name) and target.id in color_names:
                    shared_colors[target.id] = ast.literal_eval(node.value)
assert set(shared_colors) == color_names, 'Save both literal color dictionaries in Whole_integration.ipynb first.'
CELL_ANNOTATION_COLORS = shared_colors['CELL_ANNOTATION_COLORS']
DEVELOPMENTAL_STAGE_COLORS = shared_colors['DEVELOPMENTAL_STAGE_COLORS']
assert set(annotation_keys) <= set(CELL_ANNOTATION_COLORS), sorted(set(annotation_keys) - set(CELL_ANNOTATION_COLORS))
assert set(stage) <= set(DEVELOPMENTAL_STAGE_COLORS), sorted(set(stage) - set(DEVELOPMENTAL_STAGE_COLORS))
for palette in shared_colors.values():
    assert isinstance(palette, dict) and all(mcolors.is_color_like(c) for c in palette.values())
category_order = [key for key in CELL_ANNOTATION_COLORS if key in set(annotation_keys)]
STAGE_ORDER = sorted(set(stage), key=lambda value: float(value[1:]))
order = np.random.default_rng(RANDOM_SEED).permutation(a.n_obs)
lo, hi = xy.min(axis=0), xy.max(axis=0)
pad = np.maximum((hi - lo) * 0.055, 0.3)
limits = [(lo[i] - pad[i], hi[i] + pad[i]) for i in range(2)]

#3. =========== Common figure and label layout
def final_umap(keys, palette, title, margins=None):
    left, bottom, right, top = MARGINS if margins is None else margins
    width, height = left + PLOT_SIZE + right, bottom + PLOT_SIZE + top
    fig = plt.figure(figsize=(width, height), facecolor='white')
    ax = fig.add_axes([left / width, bottom / height, PLOT_SIZE / width, PLOT_SIZE / height])
    ax.scatter(xy[order, 0], xy[order, 1], c=[palette[key] for key in keys[order]],
               s=POINT_SIZE, alpha=POINT_ALPHA, linewidths=0, edgecolors='none')
    ax.set(xlim=limits[0], ylim=limits[1])
    ax.set_title(title, fontsize=TITLE_SIZE, pad=TITLE_PAD, loc='center')
    ax.set_xlabel('UMAP1', fontsize=AXIS_LABEL_SIZE, labelpad=AXIS_LABEL_PAD)
    ax.set_ylabel('UMAP2', fontsize=AXIS_LABEL_SIZE, labelpad=AXIS_LABEL_PAD)
    ax.tick_params(labelsize=TICK_SIZE, pad=TICK_PAD, length=TICK_LENGTH,
                   width=AXIS_WIDTH, colors='black')
    ax.set_box_aspect(1)
    ax.grid(False)
    for side in ('top', 'right'):
        ax.spines[side].set_visible(False)
    for side in ('left', 'bottom'):
        ax.spines[side].set(color='black', linewidth=AXIS_WIDTH)
    return fig, ax

def place_annotation_labels(fig, ax):
    labels = []
    for key in category_order:
        points = xy[annotation_keys == key]
        center = np.median(points, axis=0)
        anchor = points[np.argmin(np.sum((points - center) ** 2, axis=1))]
        age, state = key.split(' | ', 1)
        caption = age + '\n' + textwrap.fill(state.replace('_', ' '), LABEL_WRAP,
                    break_long_words=False, break_on_hyphens=False)
        text = ax.text(*anchor, caption, ha='center', va='center', fontsize=LABEL_SIZE,
                       color='black', linespacing=1.0, zorder=5)
        labels.append((key, text, anchor))
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    bounds = ax.get_window_extent(renderer)
    pixel_points = ax.transData.transform(xy)
    accepted, resolved = [], {}
    candidates = [np.zeros(2)] + [r * np.array([np.cos(t), np.sin(t)])
        for r in (6, 12, 18, 24, 32, 42, 55, 70, 90)
        for t in np.arange(0, 2 * np.pi, np.pi / 8)]
    indices = sorted(range(len(labels)), key=lambda i: -np.sum(annotation_keys == labels[i][0]))
    for i in indices:
        key, text, anchor = labels[i]
        original_box = text.get_window_extent(renderer)
        center = ax.transData.transform(anchor)
        offsets = [np.asarray(LABEL_OFFSETS_PT[key])] if key in LABEL_OFFSETS_PT else candidates
        scores = []
        for offset_pt in offsets:
            offset = offset_pt * fig.dpi / 72
            box = original_box.translated(*offset).expanded(1.035, 1.06)
            outside = max(bounds.x0 + 2 - box.x0, 0) + max(box.x1 - bounds.x1 + 2, 0)
            outside += max(bounds.y0 + 2 - box.y0, 0) + max(box.y1 - bounds.y1 + 2, 0)
            overlaps = sum(box.overlaps(other) for other in accepted)
            density = np.sum((pixel_points[:, 0] > box.x0) & (pixel_points[:, 0] < box.x1)
                             & (pixel_points[:, 1] > box.y0) & (pixel_points[:, 1] < box.y1))
            score = 1e6 * overlaps + 1e5 * outside + np.linalg.norm(offset_pt) + 0.12 * density
            scores.append((score, offset, box, offset_pt))
        _, offset, box, offset_pt = min(scores, key=lambda row: row[0])
        text.set_position(ax.transData.inverted().transform(center + offset))
        accepted.append(box)
        resolved[key] = np.asarray(offset_pt).tolist()
    fig.canvas.draw()
    boxes = [text.get_window_extent(renderer) for _, text, _ in labels]
    assert not any(boxes[i].overlaps(boxes[j]) for i in range(len(boxes)) for j in range(i)), 'Adjust LABEL_OFFSETS_PT for overlapping labels.'
    for (_, text, anchor), box in zip(labels, boxes):
        point = ax.transData.transform(anchor)
        edge = np.clip(point, [box.x0, box.y0], [box.x1, box.y1])
        if np.linalg.norm(point - edge) * 72 / fig.dpi > 12:
            end = ax.transData.inverted().transform(edge)
            ax.plot([anchor[0], end[0]], [anchor[1], end[1]], color='0.32',
                    lw=0.42, alpha=0.70, zorder=4)
    return resolved

#4. =========== Three peer clipping groups with editable PDF objects
def export_editable_pdf(fig, ax, path):
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    unit = 72 / fig.dpi
    width, height = fig.get_size_inches() * 72
    ext = ax.get_window_extent(renderer)
    assert np.allclose([ext.width * unit, ext.height * unit], PLOT_SIZE * 72)
    texts = [ax.title, ax.xaxis.label, ax.yaxis.label, *ax.texts]
    texts += [tick.label1 for tick in ax.xaxis.get_major_ticks()
              if limits[0][0] <= tick.get_loc() <= limits[0][1]]
    texts += [tick.label1 for tick in ax.yaxis.get_major_ticks()
              if limits[1][0] <= tick.get_loc() <= limits[1][1]]
    legend = ax.get_legend()
    if legend:
        texts += [legend.get_title(), *legend.get_texts()]
    texts = [t for t in texts if t.get_visible() and t.get_text()]
    for text in texts:
        box = text.get_window_extent(renderer)
        assert box.x0 >= 0 and box.y0 >= 0 and box.x1 <= fig.bbox.width and box.y1 <= fig.bbox.height, text.get_text()
        assert not text.get_path_effects()
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    pdf = canvas.Canvas(str(path), pagesize=(width, height), pageCompression=1,
                        initialFontName='Arial', initialFontSize=TICK_SIZE)
    pdf.setTitle(ax.get_title())

    def begin_group():
        pdf.saveState()
        rectangle = pdf.beginPath()
        rectangle.rect(0, 0, width, height)
        pdf.clipPath(rectangle, stroke=0, fill=0)

    def circle(point, radius, color, alpha):
        pdf.setFillColorRGB(*mcolors.to_rgb(color))
        pdf.setFillAlpha(alpha)
        pdf.circle(float(point[0]), float(point[1]), float(radius), stroke=0, fill=1)

    def line(points, color='black', linewidth=AXIS_WIDTH, alpha=1):
        pdf.setStrokeColorRGB(*mcolors.to_rgb(color))
        pdf.setStrokeAlpha(alpha)
        pdf.setLineWidth(linewidth)
        path = pdf.beginPath()
        path.moveTo(*points[0])
        for point in points[1:]:
            path.lineTo(*point)
        pdf.drawPath(path, stroke=1, fill=0)

    begin_group()  # Scatter: independent circle paths, including legend dots
    for collection in ax.collections:
        points = collection.get_offset_transform().transform(collection.get_offsets()) * unit
        faces = collection.get_facecolors()
        radius = np.sqrt(collection.get_sizes()[0]) / 2
        for i, point in enumerate(points):
            face = faces[i if len(faces) > 1 else 0]
            circle(point, radius, face[:3], float(face[3]))
    legend = ax.get_legend()
    if legend:
        for handle in legend.legend_handles:
            points = handle.get_transform().transform(handle.get_xydata()) * unit
            circle(points[len(points) // 2], handle.get_markersize() / 2,
                   handle.get_color(), POINT_ALPHA)
    pdf.restoreState()

    begin_group()  # Axes: stroked paths with editable endpoints
    for side in ('left', 'bottom'):
        spine = ax.spines[side]
        points = spine.get_transform().transform(spine.get_path().vertices) * unit
        line(points)
    for value in ax.get_xticks():
        if limits[0][0] <= value <= limits[0][1]:
            point = ax.transData.transform([value, limits[1][0]]) * unit
            line([point, point + [0, -TICK_LENGTH]])
    for value in ax.get_yticks():
        if limits[1][0] <= value <= limits[1][1]:
            point = ax.transData.transform([limits[0][0], value]) * unit
            line([point, point + [-TICK_LENGTH, 0]])
    for artist in ax.lines:
        points = artist.get_transform().transform(artist.get_xydata()) * unit
        line(points, artist.get_color(), artist.get_linewidth(), artist.get_alpha() or 1)
    pdf.restoreState()

    begin_group()  # Text: embedded Arial with real PDF text operators
    pdf.setFillColorRGB(0, 0, 0)
    for artist in texts:
        origin = artist.get_transform().transform(artist.get_position())
        angle = np.deg2rad(artist.get_rotation())
        cosine, sine = float(np.cos(angle)), float(np.sin(angle))
        for value, _, x, y in artist._get_layout(renderer)[1]:
            if not value:
                continue
            point = (origin + [x, y]) * unit
            text = pdf.beginText()
            text.setFont('Arial', artist.get_fontsize())
            text.setTextTransform(cosine, sine, -sine, cosine, *point)
            text.textOut(value)
            pdf.drawText(text)
    pdf.restoreState()
    pdf.showPage()
    pdf.save()

#5. =========== Cell annotation UMAP
fig, ax = final_umap(annotation_keys, CELL_ANNOTATION_COLORS, 'Embryonic: Cell annotation')
resolved_label_offsets = place_annotation_labels(fig, ax)
# Set the output file path.
export_editable_pdf(fig, ax, ROOT)
plt.show()
plt.close(fig)
assert np.array_equal(xy, a.obsm['X_umap'])


In [ ]:
#6. =========== Stage distribution UMAP
STAGE_LEGEND_TITLE = 'Stage (cells)'

fig, ax = final_umap(stage, DEVELOPMENTAL_STAGE_COLORS, 'Embryonic: Stage distribution')
handles = [Line2D([], [], marker='o', linestyle='', color=DEVELOPMENTAL_STAGE_COLORS[key],
                   markersize=LEGEND_MARKER_SIZE, markeredgewidth=0, alpha=POINT_ALPHA,
                   label=f'{key} ({np.sum(stage == key):,})') for key in STAGE_ORDER]
ax.legend(handles=handles, title=STAGE_LEGEND_TITLE, loc='center left',
           bbox_to_anchor=(1.025, 0.5), frameon=False, fontsize=LEGEND_SIZE,
           title_fontsize=LEGEND_SIZE, handletextpad=0.35)
# Set the output file path.
export_editable_pdf(fig, ax, ROOT)
plt.show()
plt.close(fig)
assert np.array_equal(xy, a.obsm['X_umap'])


In [ ]:
#7. =========== Cell annotation UMAP with a complete legend
ANNOTATION_LEGEND_RIGHT = 3.50  # Right margin in inches; the main axes stay 5.6 x 5.6
ANNOTATION_LEGEND_WRAP = 50     # Wrap full names without shortening or renaming
ANNOTATION_LEGEND_SPACING = 0.40

legend_margins = (MARGINS[0], MARGINS[1], ANNOTATION_LEGEND_RIGHT, MARGINS[3])
fig, ax = final_umap(annotation_keys, CELL_ANNOTATION_COLORS,
                     'Embryonic: Cell annotation', margins=legend_margins)
annotation_legend_names = [
    f'{key} ({np.sum(annotation_keys == key):,})' for key in category_order
]
handles = [
    Line2D([], [], marker='o', linestyle='', color=CELL_ANNOTATION_COLORS[key],
           markersize=LEGEND_MARKER_SIZE, markeredgewidth=0, alpha=POINT_ALPHA,
           label=textwrap.fill(name, ANNOTATION_LEGEND_WRAP,
                               break_long_words=False, break_on_hyphens=False))
    for key, name in zip(category_order, annotation_legend_names)
]
legend = ax.legend(handles=handles, loc='center left', bbox_to_anchor=(1.025, 0.5),
                    frameon=False, fontsize=LEGEND_SIZE, handletextpad=0.35,
                    labelspacing=ANNOTATION_LEGEND_SPACING)
assert len(legend.texts) == len(category_order)
assert sum(np.sum(annotation_keys == key) for key in category_order) == a.n_obs
assert not ax.texts and not ax.lines
# Set the output file path.
export_editable_pdf(fig, ax, ROOT)
plt.show()
plt.close(fig)
assert np.array_equal(xy, a.obsm['X_umap'])


In [ ]:
#8. =========== Stage distribution UMAP without a legend
fig, ax = final_umap(stage, DEVELOPMENTAL_STAGE_COLORS, 'Embryonic: Stage distribution')
assert ax.get_legend() is None and not ax.texts and not ax.lines
# Set the output file path.
export_editable_pdf(fig, ax, ROOT)
plt.show()
plt.close(fig)
assert np.array_equal(xy, a.obsm['X_umap'])
